# AVFORGE — training on Colab

Runs the **same package** as the Mac, with the same configs. Nothing is
reimplemented here: this notebook clones, installs, stages data and calls
`ddetect.train`. If a result cannot be reproduced by running those commands in
a terminal, something is wrong with the notebook, not the package.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. On CPU this
is roughly 30× slower and not worth starting.


## 1. Environment

The GPU check is not ceremony — `device_report()['authoritative']` is `False`
on anything but CUDA, and only CUDA runs are allowed to produce numbers that
go in the paper (the known-issues list).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Clone and install. Use a read-only deploy key or a public repo; do not
# paste a personal access token into a notebook that may be shared.
REPO = "https://github.com/<your-org>/deepfake-detection.git"
BRANCH = "main"

import os, pathlib
if not pathlib.Path("/content/deepfake-detection").exists():
    !git clone --depth 1 -b $BRANCH $REPO /content/deepfake-detection
%cd /content/deepfake-detection
!git log --oneline -1

In [ ]:
# Colab ships torch already; installing it again wastes several minutes and
# can downgrade the CUDA build. Install everything else.
!pip install -q -e ".[face]" --no-deps
!pip install -q timm albumentations librosa soundfile pyarrow hydra-core omegaconf \
    facenet-pytorch mediapipe pydantic rich 2>&1 | tail -2
import ddetect
from ddetect.utils.device import device_report
print(device_report())

## 2. Data

Google Drive is slow with many small files, so the cache is **tarred on Drive
and unpacked to local disk** at session start. Preprocessing 1,000 FF++ videos
directly against a mounted Drive takes hours; the same work against
`/content` takes minutes.

Run the preprocessing step **once** on any machine, tar `processed/`, and from
then on every session just unpacks it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/avforge"      # manifests + cache shards live here
!mkdir -p {DRIVE}/manifests {DRIVE}/cache {DRIVE}/runs
!ls -la {DRIVE}

In [ ]:
# Unpack the preprocessing cache to LOCAL disk.
import time, pathlib
t0 = time.time()
!mkdir -p /content/processed
for shard in sorted(pathlib.Path(f"{DRIVE}/cache").glob("*.tar")):
    print("unpacking", shard.name)
    !tar -xf {shard} -C /content/processed
print(f"{time.time()-t0:.0f}s")
!du -sh /content/processed 2>/dev/null; ls /content/processed | head

In [ ]:
# Copy the manifests across (small, so Drive is fine).
!cp {DRIVE}/manifests/*.parquet data/manifests/ 2>/dev/null || echo "no manifests on Drive yet"
!ls -la data/manifests/

from ddetect.data.manifest_io import read_manifest, summarise
import glob
for m in sorted(glob.glob("data/manifests/*.parquet")):
    print("\n==", m)
    print(summarise(read_manifest(m)))

### If the cache does not exist yet

Run this once, then tar the result to Drive so no later session repeats it.
Expect roughly 20–40 minutes per 1,000 videos with 4 workers.

In [ ]:
# ONE-TIME. Needs the raw dataset available locally or mounted.
RAW_ROOT = "/content/drive/MyDrive/datasets/FaceForensics++"

# !python -m ddetect.data.build_manifest --dataset ffpp --root {RAW_ROOT} \
#     --out data/manifests/ffpp.parquet --audit
# !python -m ddetect.data.run_preprocess --manifest data/manifests/ffpp.parquet \
#     --cache-root /content/processed --workers 4 --device cuda --phash-audit
# !cd /content && tar -cf {DRIVE}/cache/ffpp.tar processed/ffpp
# !cp data/manifests/ffpp.parquet {DRIVE}/manifests/

## 3. The leakage gate

**Run this before training, every time.** The project's headline claim is a
cross-dataset accuracy drop. If identities or near-duplicate videos cross a
split boundary, the drop shrinks for reasons unrelated to generalisation — and
the training curves still look perfectly healthy.

If this cell fails, stop. Do not train.

In [ ]:
from ddetect.data.integrity import audit
from ddetect.data.manifest_io import read_manifest

df = read_manifest("data/manifests/ffpp.parquet")
rep = audit(df, check_paths=False)
print(rep.render())
assert rep.ok, "LEAKAGE DETECTED — fix the splits before training"


## 4. Train

Each run writes `runs/<exp>/seed<N>/` with the config, git sha, environment
lock, metrics and predictions, so the result is traceable to exactly what
produced it.

Colab disconnects. The run directory is checkpointed, so re-running the same
cell resumes rather than starting over — but copy `runs/` to Drive at the end
of every session or you will lose it.

In [ ]:
EXP   = "baseline_a"      # baseline_a | baseline_b | v_full | av_proposed | full_system
MODEL = "baseline"        # baseline | v_full | avforge | avforge_wavlm
SEED  = 0
EPOCHS = 12

!python -m ddetect.train \
    --exp {EXP} --model {MODEL} --seed {SEED} --epochs {EPOCHS} \
    --manifest data/manifests/ffpp.parquet \
    --cache-root /content/processed \
    --batch-size 16 --n-frames 32 --image-size 299 \
    --num-workers 2 --freeze-epochs 2

In [ ]:
# Evaluate on the SOURCE VAL split. Target test sets are evaluated once, on a
# frozen config, through `make eval-final` -- not here (the test-set firewall).
!python -m ddetect.evaluate --run runs/{EXP}/seed{SEED} --bootstrap 2000

In [ ]:
# Back up to Drive BEFORE the runtime disconnects.
!mkdir -p {DRIVE}/runs/{EXP}
!cp -r runs/{EXP}/seed{SEED} {DRIVE}/runs/{EXP}/
!ls -la {DRIVE}/runs/{EXP}/seed{SEED}

## 5. Training curves

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in (pathlib.Path(f"runs/{EXP}/seed{SEED}/metrics.jsonl")).read_text().splitlines() if l.strip()]
rows = [r for r in rows if "epoch" in r]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot([r["epoch"] for r in rows], [r["train_loss"] for r in rows], marker="o")
ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].grid(alpha=.3)

v = [r for r in rows if "val_auc" in r]
ax[1].plot([r["epoch"] for r in v], [r["val_auc"] for r in v], marker="o", color="tab:green")
ax[1].axhline(0.5, ls="--", c="gray", lw=1)
ax[1].set_title("val AUC (source domain)"); ax[1].set_xlabel("epoch"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# Loss terms, if auxiliaries are enabled. A composite loss that quietly becomes
# dominated by one auxiliary looks healthy in aggregate while the main task
# stops improving.
terms = sorted({k for r in rows for k in r if k.startswith("loss_")})
if len(terms) > 2:
    plt.figure(figsize=(6, 3.2))
    for t in terms:
        plt.plot([r["epoch"] for r in rows if t in r], [r[t] for r in rows if t in r], marker=".", label=t.replace("loss_", ""))
    plt.legend(fontsize=8); plt.grid(alpha=.3); plt.title("loss terms"); plt.xlabel("epoch"); plt.show()

## 6. Running the grid

`run_matrix` prints the full experiment plan with its GPU-hour cost before
launching anything. Colab's session limits mean you will usually run one group
at a time.

In [ ]:
!python -m experiments.run_matrix --dry-run

In [ ]:
# Launch one group. Expect to re-run this across several sessions.
# !python -m experiments.run_matrix --group headline --only baseline_a baseline_b

---

### Notes

* **Three seeds** on headline rows (the experiment grid). `experiments/aggregate.py` marks
  single-seed rows with a dagger in the LaTeX table so a reader is never misled
  about how much evidence is behind a number.
* **Kaggle** gives 30 free GPU-hours/week versus Colab's variable quota; the
  same commands work there (`notebooks/kaggle_train.ipynb`).
* **If `val_auc` is near 1.0 on epoch 1**, suspect leakage, not success.
